In [6]:
import os
import xarray as xr
import numpy as np

In [10]:
# === Processing function ===
def calculate_maximum_6month_mean_new(start_year, end_year, monthly_mda8, monthly_temp):
    years = list(range(start_year, end_year + 1))

    max_vals = []
    T6Ms = []

    for year in years:
        # Define window: Jan of this year to Mar of next year
        start = f"{year}-01"
        end = f"{year + 1}-03"

        # Subset to this window
        subset = monthly_mda8.sel(time=slice(start, end))

        # Compute 6-month rolling mean along time
        rolling_6m = subset.rolling(time=6, center=False).mean()

        # Find index of maximum
        max_idx = rolling_6m.argmax(dim="time")
        max_val = rolling_6m.isel(time=max_idx)

        # Extract the temperature of the max 6-month period
        temp_6mon = []
        for i in range(6):
            temp_6mon.append(monthly_temp.isel(time=(max_idx-i)))
        T6M = xr.concat(temp_6mon, dim=xr.DataArray(np.arange(6), dims="month", name="month")).mean("month")

        # Expand dimensions for consistent output
        max_val = max_val.expand_dims(year=[year])
        T6M = T6M.expand_dims(year=[year])

        max_vals.append(max_val)
        T6Ms.append(T6M)

    # Combine across years
    annual_max_6m = xr.concat(max_vals, dim="year")
    temp_6month_means = xr.concat(T6Ms, dim="year")

    return annual_max_6m, temp_6month_means

In [12]:
# === Path config ===
BASE_DIR = "/glade/work/awells/air_quality/CESM/MDA8/"
TEMP_DIR = "/glade/work/awells/air_quality/CESM/TREFHT/"
SAVE_DIR = "/glade/work/awells/air_quality/CESM/OSDMA8_March/"
SCENARIOS = ["ARISE", "SSP245"]


# === Main loop ===
for scenario in SCENARIOS:
    for ens_num in range(1, 11):
        print(f"Processing {scenario}, Ensemble {ens_num:02d}")
        if scenario == "ARISE":
            o3_dates = "20350101-20691231"
            temp_dates = "203501-206912"
        else:
            o3_dates = "20200101-20691231"
            temp_dates = "202001-206912"
        file_list = [f"{BASE_DIR}MDA8_CESM2_{scenario}_{ens_num:02d}_{o3_dates}.nc"]
        temp_file = [f"{TEMP_DIR}TREFHT_CESM2_{scenario}_{ens_num:02d}_{temp_dates}.nc"]
        OSDMA8 = []
        T6M = []

        for file in file_list:
            if not os.path.exists(file):
                print(f"Missing: {file}")
                continue

            print(f"Reading {os.path.basename(file)}")
            monthly_mda8 = xr.open_dataarray(file)

            print(f"Reading {os.path.basename(temp_file[0])}")
            monthly_temp = xr.open_dataarray(temp_file[0])

            if scenario == "SSP245":
                monthly_mda8 = monthly_mda8.sel(time=slice("2020", "2069"))
                monthly_temp = monthly_temp.sel(time=slice("2020", "2069"))

            # Create list of years to calculate over
            start_year = int(str(monthly_mda8.time.dt.year[0].values))
            end_year = int(str(monthly_mda8.time.dt.year[-1].values))  # final year will be 12 months rather than 15

            annual_max_6m, temp_6month_means = calculate_maximum_6month_mean_new(start_year, end_year, monthly_mda8, monthly_temp)

            OSDMA8.append(annual_max_6m)
            T6M.append(temp_6month_means)

        if OSDMA8:
            combined = xr.concat(OSDMA8, dim="year")

            out_file = f"new_OSDMA8_CESM2_{scenario}_{ens_num:02d}_{o3_dates}.nc"
            out_path = os.path.join(SAVE_DIR, out_file)

            print(f"Saving to {out_path}")
            combined.to_netcdf(out_path)

        if T6M:
            combined = xr.concat(T6M, dim="year")

            out_file = f"T6M_CESM2_{scenario}_{ens_num:02d}_{temp_dates}.nc"
            out_path = os.path.join(SAVE_DIR, out_file)

            print(f"Saving to {out_path}")
            combined.to_netcdf(out_path)

print("All processing complete.")

Processing ARISE, Ensemble 01
Reading MDA8_CESM2_ARISE_01_20350101-20691231.nc
Reading TREFHT_CESM2_ARISE_01_203501-206912.nc
Saving to /glade/work/awells/air_quality/CESM/OSDMA8_March/new_OSDMA8_CESM2_ARISE_01_20350101-20691231.nc
Saving to /glade/work/awells/air_quality/CESM/OSDMA8_March/T6M_CESM2_ARISE_01_203501-206912.nc
Processing ARISE, Ensemble 02
Reading MDA8_CESM2_ARISE_02_20350101-20691231.nc
Reading TREFHT_CESM2_ARISE_02_203501-206912.nc
Saving to /glade/work/awells/air_quality/CESM/OSDMA8_March/new_OSDMA8_CESM2_ARISE_02_20350101-20691231.nc
Saving to /glade/work/awells/air_quality/CESM/OSDMA8_March/T6M_CESM2_ARISE_02_203501-206912.nc
Processing ARISE, Ensemble 03
Reading MDA8_CESM2_ARISE_03_20350101-20691231.nc
Reading TREFHT_CESM2_ARISE_03_203501-206912.nc
Saving to /glade/work/awells/air_quality/CESM/OSDMA8_March/new_OSDMA8_CESM2_ARISE_03_20350101-20691231.nc
Saving to /glade/work/awells/air_quality/CESM/OSDMA8_March/T6M_CESM2_ARISE_03_203501-206912.nc
Processing ARISE, E

In [4]:
# === Processing function ===
def calculate_maximum_6month_mean(start_year, end_year, monthly_mda8):
    years = list(range(start_year, end_year + 1))

    max_vals = []
    max_times = []

    for year in years:
        # Define window: Jan of this year to Mar of next year
        start = f"{year}-01"
        end = f"{year + 1}-03"

        # Subset to this window
        subset = monthly_mda8.sel(time=slice(start, end))

        # Compute 6-month rolling mean along time
        rolling_6m = subset.rolling(time=6, center=False).mean()

        # Find index of maximum
        max_idx = rolling_6m.argmax(dim="time")
        max_val = rolling_6m.isel(time=max_idx)

        # Extract the start time of the max 6-month period in month number (Jan=1, Feb=2...)
        max_time = (max_idx + 1) - 5  # +1 to account for the indexing, -5 to find start month including end month

        # Expand dimensions for consistent output
        max_val = max_val.expand_dims(year=[year])
        max_time = max_time.expand_dims(year=[year])

        max_vals.append(max_val)
        max_times.append(max_time)

    # Combine across years
    annual_max_6m = xr.concat(max_vals, dim="year")
    max_start_times = xr.concat(max_times, dim="year")

    return annual_max_6m, max_start_times

In [6]:
# === Path config ===
BASE_DIR = "/glade/work/awells/air_quality/CESM/MDA8/"
TEMP_DIR = "/glade/work/awells/air_quality/CESM/TREFHT/"
SAVE_DIR = "/glade/work/awells/air_quality/CESM/OSDMA8_March/"
SCENARIOS = ["ARISE", "SSP245"]


# === Main loop ===
for scenario in SCENARIOS:
    for ens_num in range(1, 11):
        print(f"Processing {scenario}, Ensemble {ens_num:02d}")
        if scenario == "ARISE":
            dates = "20350101-20691231"
        else:
            dates = "20200101-20691231"
        file_list = [f"{BASE_DIR}MDA8_CESM2_{scenario}_{ens_num:02d}_{dates}.nc"]
        temp_file = [f"{TEMP_DIR}MDA8_CESM2_{scenario}_{ens_num:02d}_{dates}.nc"]
        OSDMA8 = []
        OSDMA8_month = []

        for file in file_list:
            if not os.path.exists(file):
                print(f"Missing: {file}")
                continue

            print(f"Reading {os.path.basename(file)}")
            monthly_mda8 = xr.open_dataarray(file)

            print(f"Reading {os.path.basename(temp_file[0])}")
            monthly_temp = xr.open_dataarray(temp_file[0])

            # Create list of years to calculate over
            start_year = int(str(monthly_mda8.time.dt.year[0].values))
            end_year = int(str(monthly_mda8.time.dt.year[-1].values))  # final year will be 12 months rather than 15

            annual_max_6m, max_start_times = calculate_maximum_6month_mean_new(start_year, end_year, monthly_mda8, monthly_temp)

            OSDMA8.append(annual_max_6m)
            OSDMA8_month.append(max_start_times)

        if OSDMA8:
            combined = xr.concat(OSDMA8, dim="year")

            out_file = f"OSDMA8_CESM2_{scenario}_{ens_num:02d}_{dates}.nc"
            out_path = os.path.join(SAVE_DIR, out_file)

            print(f"Saving to {out_path}")
            combined.to_netcdf(out_path)

        if OSDMA8_month:
            combined = xr.concat(OSDMA8_month, dim="year")

            out_file = f"OSDMA8_startmonth_CESM2_{scenario}_{ens_num:02d}_{dates}.nc"
            out_path = os.path.join(SAVE_DIR, out_file)

            print(f"Saving to {out_path}")
            combined.to_netcdf(out_path)

print("All processing complete.")

Processing ARISE, Ensemble 01
Reading MDA8_CESM2_ARISE_01_20350101-20691231.nc
Saving to /glade/work/awells/air_quality/CESM/OSDMA8_March/OSDMA8_CESM2_ARISE_01_20350101-20691231.nc
Saving to /glade/work/awells/air_quality/CESM/OSDMA8_March/OSDMA8_startmonth_CESM2_ARISE_01_20350101-20691231.nc
All processing complete.
